# Generating test datasets

In [4]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [5]:
def _add_message(messages: list, role: str, text: str) -> None:
    message = {"role": role, "content": text}
    messages.append(message)

def add_user_message(messages: list, text: str) -> None:
    _add_message(messages, "user", text)

def add_assistant_message(messages: list, text: str) -> None:
    _add_message(messages, "assistant", text)

def chat(messages: list, system_prompt: str | None = None, temperature: float=1.0, stop: list | None = None) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system_prompt is not None:
        params["system"] = system_prompt
    if stop is not None:
        params["stop_sequences"] = stop
    response = client.messages.create(
        **params
    )
    return response.content[0].text


In [6]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop=["```"])
    return json.loads(text)

In [ ]:
import pprint
dataset = generate_dataset()
pprint.pprint(dataset)

[{'task': 'Write a Python function that extracts the AWS region from an S3 '
          "bucket URI in the format 's3://bucket-name/key'. Return 'us-east-1' "
          'if no region is explicitly specified.'},
 {'task': 'Create a JSON CloudFormation template snippet that defines an AWS '
          'Lambda function resource with basic properties: FunctionName, '
          'Runtime (python3.11), Handler, and Role.'},
 {'task': 'Write a regex pattern that matches valid AWS IAM role ARNs in the '
          "format 'arn:aws:iam::123456789012:role/RoleName' and captures the "
          'account ID and role name separately.'}]


In [8]:
with open('artifacts/dataset.json', 'w') as file_path:
    json.dump(dataset, file_path, indent=2)

# Running the eval

In [ ]:
def run_prompt(test_case: dict) -> str:
    # Merges the prompt and test case input, the returns the result
    prompt = f"""
Please solve the following task:

{test_case['task']}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary, or explanations.
"""
    messages = []
    add_user_message(messages, prompt)
    #add_assistant_message(messages, "```")
    text = chat(messages) #, stop=["```"])
    return text

def run_test_case(test_case: dict) -> dict:
    # Calls run_prompt, then grades the result
    output = run_prompt(test_case)

    # TODO: Grading
    score = 10
    
    return {
        "output": output, 
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset: list) -> list:
    # Loops through the dataset, running each test case and grading it
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [10]:
with open('artifacts/dataset.json', 'r') as file_path:
    dataset = json.load(file_path)

results = run_eval(dataset)

In [11]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Region Extraction from S3 URI\n\nHere's a solution with multiple approaches:\n\n## Approach 1: Simple String Parsing (Basic)\n\n```python\ndef extract_region_from_s3_uri(s3_uri: str) -> str:\n    \"\"\"\n    Extract AWS region from an S3 bucket URI.\n    \n    Args:\n        s3_uri: S3 URI in format 's3://bucket-name/key' or 's3://bucket-name'\n    \n    Returns:\n        AWS region string, defaults to 'us-east-1' if not specified\n    \"\"\"\n    # Basic URIs don't contain region info - they default to us-east-1\n    return 'us-east-1'\n```\n\n## Approach 2: Using Boto3 (Recommended for Real AWS Usage)\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError\n\ndef extract_region_from_s3_uri(s3_uri: str) -> str:\n    \"\"\"\n    Extract AWS region from an S3 bucket URI by querying AWS.\n    \n    Args:\n        s3_uri: S3 URI in format 's3://bucket-name/key'\n    \n    Returns:\n        AWS region string, defaults to 'us-east-1' if not found\n

# Model based grading

In [12]:
def grade_by_model(test_case: dict, output: str) -> dict:
    # Grades the output using a model
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.
    
    Original Task: 
    <task>
    {test_case['task']}
    </task>

    Solution to evaluate: 
    <solution>
    {output}
    </solution>
    
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": "string",
        "score": number
    }}
    """
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop=["```"])
    return json.loads(eval_text)

In [13]:
# Update run_test_case to use model-based grading
def run_test_case(test_case: dict) -> dict:
    # Calls run_prompt, then grades the result
    output = run_prompt(test_case)

    # TODO: Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output, 
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [14]:
with open('artifacts/dataset.json', 'r') as file_path:
    dataset = json.load(file_path)

results = run_eval(dataset)

In [15]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a solution that extracts the AWS region from an S3 bucket URI:\n\n```python\nimport boto3\nimport re\nfrom botocore.exceptions import ClientError\n\ndef extract_region_from_s3_uri(s3_uri):\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URI.\n    \n    Args:\n        s3_uri (str): S3 URI in format 's3://bucket-name/key' or 's3://bucket-name'\n    \n    Returns:\n        str: AWS region name (defaults to 'us-east-1' if not found)\n    \n    Raises:\n        ValueError: If the S3 URI format is invalid\n    \"\"\"\n    # Validate S3 URI format\n    s3_pattern = r'^s3://([a-z0-9\\.\\-]+)(/.*)?$'\n    match = re.match(s3_pattern, s3_uri)\n    \n    if not match:\n        raise ValueError(f\"Invalid S3 URI format: {s3_uri}\")\n    \n    bucket_name = match.group(1)\n    \n    try:\n        # Create S3 client and get bucket location\n        s3_client = boto3.client('s3')\n        response = s3_client.get_bucket_loc

In [16]:
from statistics import mean

# Update run_eval to create an average score for the dataset
def run_eval(dataset: list) -> list:
    # Loops through the dataset, running each test case and grading it
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")

    return results

In [18]:
with open('artifacts/dataset.json', 'r') as file_path:
    dataset = json.load(file_path)

results = run_eval(dataset)

Average Score: 6.333333333333333


In [19]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Region Extraction from S3 URI\n\nHere's a comprehensive solution:\n\n```python\nimport re\nimport boto3\nfrom botocore.exceptions import ClientError\n\ndef extract_region_from_s3_uri(s3_uri: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URI.\n    \n    Args:\n        s3_uri: S3 URI in format 's3://bucket-name/key' or 's3://bucket-name'\n    \n    Returns:\n        AWS region string (default: 'us-east-1')\n    \n    Raises:\n        ValueError: If the URI format is invalid\n    \"\"\"\n    \n    # Validate S3 URI format\n    s3_pattern = r'^s3://([a-z0-9.-]+)(?:/.*)?$'\n    match = re.match(s3_pattern, s3_uri)\n    \n    if not match:\n        raise ValueError(f\"Invalid S3 URI format: {s3_uri}\")\n    \n    bucket_name = match.group(1)\n    \n    # Try to get the bucket's region from AWS\n    try:\n        s3_client = boto3.client('s3', region_name='us-east-1')\n        response = s3_client.get_bucket_location(Bucket=bucket_name)\n  

# Code based grading

In [20]:
import ast
import re

def validate_json(text: str) -> int:
    try:
        json.loads(text)
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text: str) -> int:
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text: str) -> int:
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [26]:
def grade_syntax(test_case: dict, output: str) -> int:
    format = test_case.get("format")
    if format == "json":
        return validate_json(output)
    elif format == "python":
        return validate_python(output)
    elif format == "regex":
        return validate_regex(output)
    else:
        raise ValueError(f"Unknown format: {format}")

# Overwrite generate_dataset to include format field
def generate_dataset() -> list:
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "python" or "json" or "regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop=["```"])
    return json.loads(text)

In [22]:
dataset = generate_dataset()
pprint.pprint(dataset)

with open('artifacts/dataset_with_format.json', 'w') as file_path:
    json.dump(dataset, file_path, indent=2)

[{'format': 'regex',
  'task': 'Parse an AWS S3 bucket name from an S3 URI (e.g., '
          "'s3://my-bucket/path/to/file.txt') and extract just the bucket "
          'name'},
 {'format': 'json',
  'task': 'Create a JSON CloudFormation template snippet that defines an AWS '
          'Lambda function resource with basic properties (FunctionName, '
          'Runtime, Handler, and Role)'},
 {'format': 'python',
  'task': 'Write a Python function that takes an AWS IAM policy JSON object '
          'and returns a list of all Effect values present in the policy '
          'statements'}]


In [23]:
# overwrite run_prompt to add prefilling with ```code
def run_prompt(test_case: dict) -> str:
    # Merges the prompt and test case input, the returns the result
    prompt = f"""
Please solve the following task:

{test_case['task']}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary, or explanations.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    text = chat(messages, stop=["```"])
    return text


In [ ]:
# update the grading in run_test_case to use the new grade_syntax function
# Update run_test_case to use model-based grading
def run_test_case(test_case: dict) -> dict:
    # Calls run_prompt, then grades the result
    output = run_prompt(test_case)

    # TODO: Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_syntax(test_case, output)

    overall_score = (model_score + syntax_score) / 2
    return {
        "output": output, 
        "test_case": test_case,
        "score": overall_score,
        "reasoning": reasoning,
        "syntax_score": syntax_score,
        "model_score": model_score
    }

In [28]:
with open('artifacts/dataset_with_format.json', 'r') as file_path:
    dataset = json.load(file_path)

results = run_eval(dataset)

Average Score: 8.5


In [29]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef extract_s3_bucket(uri):\n    match = re.match(r's3://([^/]+)', uri)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket/path/to/file.txt') and extract just the bucket name",
      "format": "regex"
    },
    "score": 8.5,
    "reasoning": "The solution correctly extracts the bucket name from valid S3 URIs. However, it lacks robustness for edge cases like None inputs and doesn't validate against AWS bucket naming constraints. Using string operations instead of regex would be more efficient and readable for this straightforward parsing task. Adding basic input validation would make it production-ready.",
    "syntax_score": 10
  },
  {
    "output": "\n{\n  \"MyLambdaFunction\": {\n    \"Type\": \"AWS::Lambda::Function\",\n    \"Properties\": {\n      \"FunctionName\": \"my-function\",\n      \"Runtime\": \"python3.11\",\n      \"Handler\": \"index.ha

# Exercise on prompt evals

In [30]:
# Upgrade the generate_dataset function
def generate_dataset() -> list:
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "python" or "json" or "regex"
    "solution_criteria": "A concise description of what a correct solution should include, to help guide evaluation"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop=["```"])
    return json.loads(text)

In [31]:
dataset = generate_dataset()
pprint.pprint(dataset)

with open('artifacts/dataset_with_format.json', 'w') as file_path:
    json.dump(dataset, file_path, indent=2)

[{'format': 'regex',
  'solution_criteria': 'A regex pattern that captures the bucket name and '
                       'object key separately from a valid S3 URI, handling '
                       'paths with multiple slashes and various valid bucket '
                       'naming conventions',
  'task': 'Parse an AWS S3 bucket name and object key from an S3 URI in the '
          "format 's3://bucket-name/path/to/object'"},
 {'format': 'json',
  'solution_criteria': 'A valid IAM policy JSON with correct Action, Resource, '
                       'and Effect fields that grants s3:GetObject and '
                       's3:ListBucket permissions for a specified bucket ARN',
  'task': 'Create a JSON IAM policy document that allows an AWS principal to '
          'read and list objects in a specific S3 bucket'},
 {'format': 'python',
  'solution_criteria': 'A function that checks if the status string contains '
                       "'COMPLETE' and does not contain 'FAILED' or "
     

In [32]:
# Update the grade_by_model function to include solution_criteria in the evaluation prompt
def grade_by_model(test_case: dict, output: str) -> dict:
    # Grades the output using a model
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.
    
    Original Task: 
    <task>
    {test_case['task']}
    </task>

    Solution to evaluate: 
    <solution>
    {output}
    </solution>

    Solution criteria to guide your evaluation:
    <criteria>
    {test_case.get('solution_criteria', 'No specific criteria provided')}
    </criteria>
    
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": "string",
        "score": number
    }}
    """
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop=["```"])
    return json.loads(eval_text)

In [34]:
with open('artifacts/dataset_with_format.json', 'r') as file_path:
    dataset = json.load(file_path)

results = run_eval(dataset)

Average Score: 6.666666666666667


In [35]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\nfrom urllib.parse import urlparse\n\ndef parse_s3_uri(uri):\n    match = re.match(r'^s3://([a-z0-9.-]+)/(.+)$', uri)\n    if match:\n        return {\n            \"bucket\": match.group(1),\n            \"key\": match.group(2)\n        }\n    return None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name and object key from an S3 URI in the format 's3://bucket-name/path/to/object'",
      "format": "regex",
      "solution_criteria": "A regex pattern that captures the bucket name and object key separately from a valid S3 URI, handling paths with multiple slashes and various valid bucket naming conventions"
    },
    "score": 8.0,
    "reasoning": "The solution correctly parses valid S3 URIs for basic cases but has a flawed bucket name pattern that rejects valid bucket names (those with uppercase letters or different character combinations). The unused import and lack of comprehensive validation indicate incomplete implementation. The r